# 1eNp0pi CAF Selection: Pandora vs. Pandora+NuGraph2

Full cut-sequence efficiency/purity for both selection columns from the
table, direct from flat-CAF files (no intermediate flattening macro).

## Data source
```
/pnfs/icarus/scratch/users/sdey2/v10_06_00_04p04/nugraph/BNB_NuGraph_v10_06_00_04p04_test/caf/5000/files.list
```

## Caveats -- read before trusting a number out of this

1. **Fiducial volume cut is a PLACEHOLDER** (`FV_BOUNDS` below). No FV flag
   exists in the CAF -- checked directly, nothing like it in the branch
   list. Replace with your real cut before trusting anything.
2. **`rec.slc.truth.nproton` reads uniformly 0** on the one file this was
   built/tested against (50 events) -- meaning zero slices currently pass
   the truth-signal definition on this file. Could be real small-sample
   statistics (nue CC is rare in a numu-dominated beam) or could mean this
   variable isn't what's assumed here. Check on a bigger sample.
3. **"Electron ID" vs. "Electron cuts" are kept as separate steps**,
   matching the table -- a real bug caught while building this: an
   earlier version measured "has electron candidate" as "candidate found
   AND has valid shower energy," which silently conflated the two rows.
   Now: Electron ID = a qualifying primary PFP exists at all (should be
   close to universal); Electron cuts = that PFP's shower vars actually
   pass the energy/dE/dx/angle/gap thresholds (this is where a PFP with
   no valid shower reconstruction correctly gets rejected instead).
4. **Proton ID / no-other-particles / muon veto are now based on
   per-PFP classification** of every OTHER primary PFP (excluding the
   chosen electron), not a raw count -- Pandora: `chi2_muon`/`chi2_proton`
   thresholds; NG2: `ngscore.sem_cat`. Still simplified relative to the
   full contained/KE-threshold PID-after-electron logic in the table
   (that needs per-PFP kinetic energy and containment, not implemented
   here yet) -- flagged inline where it matters.
5. **Flat-CAF unflattening gotcha**: `rec.slc.reco.pfp.X` branches are
   flattened across slice boundaries *within each event*. Recovering
   `(event, slice, pfp)` needs the counter branch
   `rec.slc.reco.pfp..length` and a two-step `ak.unflatten` -- done once
   in `load_caf()` below.
6. **Built and tested against exactly ONE file** (50 events, 332 slices).
   Confirms mechanics work, not a statement the physics numbers are final.

In [ ]:
import uproot
import awkward as ak
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from cmcrameri import cm as cmc

plt.rcParams.update({
    'font.family': 'serif', 'font.serif': ['Times New Roman', 'DejaVu Serif'],
    'mathtext.fontset': 'cm', 'axes.labelsize': 14, 'axes.titlesize': 13,
    'figure.dpi': 140, 'figure.facecolor': 'white', 'axes.facecolor': '#f9f9f9',
    'axes.grid': True, 'grid.color': '#dddddd', 'grid.linewidth': 0.7,
})
BLUE, CRIMSON = '#3a7ebf', '#DC143C'
MANAGUA = cmc.managua

## 1. Load and flatten CAF file(s) into a per-slice DataFrame

In [ ]:
SLC_BRANCHES = [
    'rec.slc.self', 'rec.slc.is_clear_cosmic', 'rec.slc.vertex.x', 'rec.slc.vertex.y', 'rec.slc.vertex.z',
    'rec.slc.fmatch.present', 'rec.slc.nu_score',
    'rec.slc.truth.E', 'rec.slc.truth.iscc', 'rec.slc.truth.pdg',
    'rec.slc.truth.npiplus', 'rec.slc.truth.npiminus', 'rec.slc.truth.npizero', 'rec.slc.truth.nproton',
]
PFP_BRANCHES = [
    'rec.slc.reco.pfp.parent_is_primary', 'rec.slc.reco.pfp.trackScore', 'rec.slc.reco.pfp.ngscore.sem_cat',
    'rec.slc.reco.pfp.shw.bestplane_energy', 'rec.slc.reco.pfp.shw.bestplane_dEdx',
    'rec.slc.reco.pfp.shw.open_angle', 'rec.slc.reco.pfp.shw.conversion_gap',
    'rec.slc.reco.pfp.shw.plane.0.nHits', 'rec.slc.reco.pfp.shw.plane.1.nHits', 'rec.slc.reco.pfp.shw.plane.2.nHits',
    'rec.slc.reco.pfp.trk.chi2pid.2.chi2_muon', 'rec.slc.reco.pfp.trk.chi2pid.2.chi2_proton',
]

def pick_best_idx(mask, score):
    """keepdims=True index of the highest-`score` PFP passing `mask` per
    slice, None (masked) where nothing in that slice passes."""
    return ak.argmax(ak.mask(score, mask), axis=-1, keepdims=True)

def flat1(x):
    return ak.to_numpy(ak.flatten(x, axis=1))
def flat1_opt(x):
    return ak.to_numpy(ak.fill_none(ak.flatten(x, axis=1), np.nan))

def load_caf(paths, tree='recTree'):
    if isinstance(paths, str):
        paths = [paths]

    slc_parts = {b: [] for b in SLC_BRANCHES}
    pfp_parts = {b: [] for b in PFP_BRANCHES}
    truth_mult_parts = {
        'nproton': [],
        'npiplus': [],
        'npiminus': [],
        'npizero': [],
    }

    for path in paths:
        rec = uproot.open(path)[tree]
        counts = rec['rec.slc.reco.pfp..length'].array()   # (event, slice) -> n_pfp
        n_slices_per_event = ak.num(counts, axis=1)
        flat_counts = ak.flatten(counts, axis=1)

        for b in SLC_BRANCHES:
            slc_parts[b].append(rec[b].array())

        # The stored rec.slc.truth multiplicity summaries are unfilled in
        # this production. Reconstruct one value per slice from prim.pdg.
        prim_pdg = rec['rec.slc.truth.prim.pdg'].array()
        prim_len = rec['rec.slc.truth.prim..length'].array()

        derived = {
            'nproton': [],
            'npiplus': [],
            'npiminus': [],
            'npizero': [],
        }

        for event_pdgs, event_lengths in zip(prim_pdg, prim_len):
            pdgs = ak.to_numpy(event_pdgs)
            lengths = ak.to_numpy(event_lengths).astype(np.int64)

            if lengths.sum() != len(pdgs):
                raise RuntimeError(
                    f'{path}: sum(prim lengths)={lengths.sum()} '
                    f'but len(prim pdg)={len(pdgs)}'
                )

            start = 0
            for length in lengths:
                slice_pdgs = pdgs[start:start + length]
                start += length

                derived['nproton'].append(
                    np.count_nonzero(slice_pdgs == 2212)
                )
                derived['npiplus'].append(
                    np.count_nonzero(slice_pdgs == 211)
                )
                derived['npiminus'].append(
                    np.count_nonzero(slice_pdgs == -211)
                )
                derived['npizero'].append(
                    np.count_nonzero(slice_pdgs == 111)
                )

        for name in truth_mult_parts:
            truth_mult_parts[name].append(
                np.asarray(derived[name], dtype=np.int32)
            )

        for b in PFP_BRANCHES:
            flat = ak.flatten(rec[b].array(), axis=1)
            per_slice = ak.unflatten(flat, flat_counts)
            pfp_parts[b].append(ak.unflatten(per_slice, n_slices_per_event))

    slc = {b: ak.concatenate(v) for b, v in slc_parts.items()}
    pfp = {b: ak.concatenate(v) for b, v in pfp_parts.items()}
    truth_mult = {
        name: np.concatenate(parts)
        for name, parts in truth_mult_parts.items()
    }

    pfp_primary = pfp['rec.slc.reco.pfp.parent_is_primary'] == 1
    n_hits_tot = (pfp['rec.slc.reco.pfp.shw.plane.0.nHits'] +
                  pfp['rec.slc.reco.pfp.shw.plane.1.nHits'] +
                  pfp['rec.slc.reco.pfp.shw.plane.2.nHits'])
    local_idx = ak.local_index(pfp_primary, axis=-1)

    def electron_pick(e_mask):
        idx = pick_best_idx(e_mask, n_hits_tot)
        idx_scalar = ak.firsts(idx, axis=-1)          # scalar-or-None per slice
        is_e_pfp = local_idx == ak.fill_none(idx_scalar, -999)
        has_candidate = ~ak.is_none(idx_scalar, axis=1) # ELECTRON ID (candidate exists), separate from cuts
        # NOTE: axis=1 matters -- ak.is_none's default axis=0 checks the outermost
        # (event) level, not element-wise per-slice, and silently gives the wrong
        # (event-level, not slice-level) shape otherwise. Caught by testing against
        # real data before handing this over.
        return idx, is_e_pfp, has_candidate

    idx_e_p, is_e_pfp_p, has_e_id_p = electron_pick(pfp_primary & (pfp['rec.slc.reco.pfp.trackScore'] < 0.5))
    idx_e_n, is_e_pfp_n, has_e_id_n = electron_pick(pfp_primary & (pfp['rec.slc.reco.pfp.ngscore.sem_cat'] == 2))

    def gather(field, idx):
        return ak.firsts(field[idx], axis=-1)

    # --- other (non-electron) primary PFP classification, both schemes ---
    other_p = pfp_primary & ~is_e_pfp_p
    other_n = pfp_primary & ~is_e_pfp_n

    muon_like_p  = other_p & (pfp['rec.slc.reco.pfp.trk.chi2pid.2.chi2_muon'] < 30) & (pfp['rec.slc.reco.pfp.trk.chi2pid.2.chi2_proton'] > 60)
    proton_like_p = other_p & ~muon_like_p & (pfp['rec.slc.reco.pfp.trk.chi2pid.2.chi2_proton'] < 100)
    unclassified_p = other_p & ~muon_like_p & ~proton_like_p

    muon_like_n   = other_n & (pfp['rec.slc.reco.pfp.ngscore.sem_cat'] == 0)
    proton_like_n = other_n & (pfp['rec.slc.reco.pfp.ngscore.sem_cat'] == 1)
    unclassified_n = other_n & ~muon_like_n & ~proton_like_n

    df = pd.DataFrame({
        'true_nu_energy': flat1(slc['rec.slc.truth.E']),
        'iscc':     flat1(slc['rec.slc.truth.iscc']),
        'nu_pdg':   flat1(slc['rec.slc.truth.pdg']),
        'npiplus':  truth_mult['npiplus'],
        'npiminus': truth_mult['npiminus'],
        'npizero':  truth_mult['npizero'],
        'nproton':  truth_mult['nproton'],
        'is_clear_cosmic': flat1(slc['rec.slc.is_clear_cosmic']),
        'fmatch_present':  flat1(slc['rec.slc.fmatch.present']),
        'vtx_x': flat1(slc['rec.slc.vertex.x']),
        'vtx_y': flat1(slc['rec.slc.vertex.y']),
        'vtx_z': flat1(slc['rec.slc.vertex.z']),

        'has_e_id_pandora': flat1(has_e_id_p),
        'has_e_id_ng':      flat1(has_e_id_n),
        'shw_e_pandora':     flat1_opt(gather(pfp['rec.slc.reco.pfp.shw.bestplane_energy'], idx_e_p)),
        'shw_dedx_pandora':  flat1_opt(gather(pfp['rec.slc.reco.pfp.shw.bestplane_dEdx'], idx_e_p)),
        'shw_angle_pandora': flat1_opt(gather(pfp['rec.slc.reco.pfp.shw.open_angle'], idx_e_p)),
        'shw_gap_pandora':   flat1_opt(gather(pfp['rec.slc.reco.pfp.shw.conversion_gap'], idx_e_p)),
        'shw_e_ng':          flat1_opt(gather(pfp['rec.slc.reco.pfp.shw.bestplane_energy'], idx_e_n)),
        'shw_dedx_ng':       flat1_opt(gather(pfp['rec.slc.reco.pfp.shw.bestplane_dEdx'], idx_e_n)),
        'shw_angle_ng':      flat1_opt(gather(pfp['rec.slc.reco.pfp.shw.open_angle'], idx_e_n)),
        'shw_gap_ng':        flat1_opt(gather(pfp['rec.slc.reco.pfp.shw.conversion_gap'], idx_e_n)),

        'has_proton_like_pandora': flat1(ak.any(proton_like_p, axis=-1)),
        'has_muon_like_pandora':   flat1(ak.any(muon_like_p, axis=-1)),
        'n_unclassified_pandora':  flat1(ak.sum(unclassified_p, axis=-1)),
        'has_proton_like_ng':      flat1(ak.any(proton_like_n, axis=-1)),
        'has_muon_like_ng':        flat1(ak.any(muon_like_n, axis=-1)),
        'n_unclassified_ng':       flat1(ak.sum(unclassified_n, axis=-1)),
    })
    for name, values in truth_mult.items():
        if len(values) != len(df):
            raise RuntimeError(
                f'Derived {name} has {len(values)} entries, '
                f'but the DataFrame has {len(df)} slices'
            )

    return df.replace(-5.0, np.nan)

# EDIT ME: path(s) to your CAF file(s) -- single file for a quick check,
# or read files.list (or a slice of it) for real statistics.
CAF_PATHS = ['/exp/icarus/app/users/sdey2/icaruscode-v10_06_00_04p04_production/nugraphProduction/fermigridJobScripts/stage0_run2_wcdnn_icarus_overlay_82703517_6971-21c72645-9a7c-4161-bef2-41d68f3acd25_20260706T083027-MCstage1.flat.caf.root']
#CAF_PATHS = ['/pnfs/icarus/scratch/users/sdey2/v10_06_00_04p04/nugraph/BNB_NuGraph_v10_06_00_04p04_test/caf/5000/files.list']

df = load_caf(CAF_PATHS)
print(f'Slices: {len(df):,}')
print(f'Electron ID candidates -- Pandora: {df.has_e_id_pandora.sum():,}   NG2: {df.has_e_id_ng.sum():,}')
print(f'...of which valid shower energy  -- Pandora: {df.shw_e_pandora.notna().sum():,}   NG2: {df.shw_e_ng.notna().sum():,}')

## 2. Truth topology classification

Categories: signal (nue CC, 0pi, >=1p), other nue CC, numu CC, NC, and
cosmic/no-truth-match (`nu_pdg` sentinel, e.g. -1). Not fancy -- just
enough to see what's actually in the sample before selecting on it.

In [ ]:
def classify_topology(df):
    is_nue_cc  = (df.nu_pdg.abs() == 12) & (df.iscc == 1)
    is_numu_cc = (df.nu_pdg.abs() == 14) & (df.iscc == 1)
    is_nc      = (df.iscc == 0) & df.nu_pdg.abs().isin([12, 14])
    is_signal  = is_nue_cc & (df.npiplus==0) & (df.npiminus==0) & (df.npizero==0) & (df.nproton>=1)
    is_cosmic  = ~df.nu_pdg.abs().isin([12, 14])   # no true neutrino match at all

    topo = np.full(len(df), 'other', dtype=object)
    topo[is_nc.to_numpy()]      = r'NC'
    topo[is_numu_cc.to_numpy()] = r'$\nu_\mu$ CC'
    topo[is_nue_cc.to_numpy()]  = r'$\nu_e$ CC (other)'
    topo[is_signal.to_numpy()]  = r'$\nu_e$ CC 0$\pi$Np (signal)'
    topo[is_cosmic.to_numpy()]  = 'Cosmic / no truth'
    return pd.Series(topo, index=df.index), is_signal

df['topology'], sig = classify_topology(df)
print(df['topology'].value_counts())

order = [r'$\nu_e$ CC 0$\pi$Np (signal)', r'$\nu_e$ CC (other)', r'$\nu_\mu$ CC', r'NC', 'Cosmic / no truth']
colors = [MANAGUA(x) for x in np.linspace(0.05, 0.95, len(order))]

fig, ax = plt.subplots(figsize=(9, 5.5))
bins = np.linspace(0, 4, 21)
stack_data = [df.loc[df.topology==t, 'true_nu_energy'].dropna() for t in order]
ax.hist(stack_data, bins=bins, stacked=True, color=colors, label=order)
ax.set_xlabel(r'$E_\nu^{\mathrm{true}}$ [GeV]')
ax.set_ylabel('Slices')
ax.set_title('Truth topology breakdown')
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('caf_topology_breakdown.png', dpi=150)
plt.show()

## 3. Direct check: does `rec.slc.ng_plane` (slice-level NuGraph info) actually exist in the file?

Separate from `rec.slc.reco.pfp.ngscore.sem_cat` checked above -- `ng_plane`
is a different branch (`SRNuGraphSliceInfo ng_plane[3]` on `SRSlice`, one
struct per wire plane) added on the `feature/sdey_NuGraph2_MoveVars_v04p04`
sbnanaobj branch. This is the field `cafe` couldn't compile against tonight
(`no member named 'ng_plane' in 'caf::Proxy<caf::SRSlice>'`) because the
`sbnana_v04p04` analysis environment was built against stock CVMFS
`sbnanaobj v10_00_05`, which doesn't declare it -- but that's a read-side
C++ class problem, not a "does the data exist" problem. `uproot` reads raw
branch bytes regardless of what any compiled class knows about, so this
checks the file directly, no `cafe`/mrb environment needed.

In [ ]:
f = uproot.open(CAF_PATHS[0])
rec = f['recTree']
keys = rec.keys()

ng_plane_branches = [k for k in keys if k.startswith('rec.slc.ng_plane')]
print(f'Found {len(ng_plane_branches)} rec.slc.ng_plane branches:')
for b in ng_plane_branches:
    print(' ', b)

if not ng_plane_branches:
    print()
    print('No rec.slc.ng_plane branches found at all -- would mean this specific')
    print('field genuinely did not make it into this CAF (different conclusion')
    print('from the ngscore.sem_cat check above -- worth flagging if this happens).')
else:
    print()
    # spot-check a couple of the actual fields that failed to compile tonight
    for field in ['shr_hits', 'ng_vtx_hip_hits', 'unclustered_shr_hits']:
        for plane in [0, 1, 2]:
            branch = f'rec.slc.ng_plane.{plane}.{field}'
            if branch in keys:
                arr = ak.flatten(rec[branch].array(), axis=1)
                arr_np = ak.to_numpy(arr)
                n_valid = np.sum(~np.isnan(arr_np) & (arr_np != -5.0))
                print(f'{branch}: {len(arr_np)} slices, {n_valid} non-sentinel values, '
                      f'mean={np.nanmean(arr_np[arr_np != -5.0]) if n_valid else float("nan"):.2f}')
            else:
                print(f'{branch}: NOT FOUND')